In [ ]:
orders = data["orders"]
shipped = orders[orders["Is Shipped"]]

# RTO tags weren't consistently applied before this month (RTO rate is
# ~0% for every earlier month) -- exclude that period from RTO-rate
# calculations so older orders don't look falsely 100% delivered.
rto_reliable = shipped["Order Month"] >= RTO_TRACKING_START if "Order Month" in shipped.columns else shipped.index >= 0
shipped_rto = shipped[rto_reliable]

delivery_rate = shipped_rto["Is Delivered"].mean() * 100
rto_rate = shipped_rto["Any RTO"].mean() * 100
total_rto = orders["Any RTO"].sum()
rto_of_total = orders["Any RTO"].mean() * 100

st.subheader("Delivery Performance Overview")
st.caption(
    f"Delivery/RTO rates below are computed on orders shipped from {RTO_TRACKING_START} onward "
    "(RTO Initiated/Delivered tags weren't consistently applied before then, which would otherwise "
    "make the delivery rate look artificially close to 100%). 'Total RTO Orders' and 'RTO % of All "
    "Orders' still cover the full history."
)
c1, c2, c3, c4 = st.columns(4)
with c1:
    kpi_card("Delivery Rate (Shipped)", f"{delivery_rate:.2f}%")
with c2:
    kpi_card("RTO Rate (Shipped)", f"{rto_rate:.2f}%", positive=False)
with c3:
    kpi_card("Total RTO Orders", f"{total_rto:,.0f}")
with c4:
    kpi_card("RTO % of All Orders", f"{rto_of_total:.2f}%", positive=False)

In [ ]:
col1, col2 = st.columns([1, 1.4])
with col1:
    pie = pd.DataFrame({
        "Status": ["Delivered", "RTO"],
        "Count": [shipped_rto["Is Delivered"].sum(), shipped_rto["Any RTO"].sum()],
    })
    fig = px.pie(pie, names="Status", values="Count", hole=0.55,
                  color="Status", color_discrete_map={"Delivered": SUCCESS, "RTO": DANGER},
                  title="Delivered vs RTO (Shipped Orders)")
    st.plotly_chart(fig, width='stretch')

with col2:
    if "Order Month" in orders.columns:
        agg_kwargs = dict(
            Total=("Name", "count"),
            Shipped=("Is Shipped", "sum"),
            RTO=("Any RTO", "sum"),
        )
        if "RTO Initiated" in orders.columns:
            agg_kwargs["RTO Initiated Count"] = ("RTO Initiated", "sum")
        if "RTO Delivered" in orders.columns:
            agg_kwargs["RTO Delivered Count"] = ("RTO Delivered", "sum")

        trend = orders.groupby("Order Month").agg(**agg_kwargs).reset_index()
        trend = trend[trend["Shipped"] > 0]
        trend["RTO % of Shipped"] = trend["RTO"] / trend["Shipped"] * 100
        trend["RTO % of All Orders"] = trend["RTO"] / trend["Total"] * 100
        trend["Delivered % of Shipped"] = 100 - trend["RTO % of Shipped"]
        trend["Order Month"] = trend["Order Month"].astype(str)

        fig = go.Figure()
        fig.add_scatter(x=trend["Order Month"], y=trend["Delivered % of Shipped"], mode="lines+markers",
                         name="Delivered % of Shipped", line=dict(color=SUCCESS))
        fig.add_scatter(x=trend["Order Month"], y=trend["RTO % of Shipped"], mode="lines+markers",
                         name="RTO % of Shipped (Undelivered)", line=dict(color=DANGER))
        if "RTO Initiated Count" in trend.columns:
            trend["RTO Initiated % of Shipped"] = trend["RTO Initiated Count"] / trend["Shipped"] * 100
            fig.add_scatter(x=trend["Order Month"], y=trend["RTO Initiated % of Shipped"], mode="lines+markers",
                             name="RTO Initiated % (in transit/return)", line=dict(color=ACCENT1, dash="dot"))
        if "RTO Delivered Count" in trend.columns:
            trend["RTO Delivered % of Shipped"] = trend["RTO Delivered Count"] / trend["Shipped"] * 100
            fig.add_scatter(x=trend["Order Month"], y=trend["RTO Delivered % of Shipped"], mode="lines+markers",
                             name="RTO Delivered % (returned to origin)", line=dict(color=PALETTE[4], dash="dot"))
        fig.add_scatter(x=trend["Order Month"], y=trend["RTO % of All Orders"], mode="lines+markers",
                         name="RTO % of All Orders", line=dict(color=ACCENT2, dash="dash"))
        fig.update_layout(title="Monthly Delivery / RTO Breakdown (%)", yaxis_title="% of Shipped Orders")
        st.plotly_chart(fig, width='stretch')
        st.caption(
            "'RTO Initiated' = return is in progress (courier marked undeliverable / customer refused); "
            "'RTO Delivered' = the returned shipment has made it back to origin. Both count as undelivered."
        )

In [ ]:
st.subheader("Delivery Success Rate by Geography")
st.caption(
    f"Limited to orders shipped from {RTO_TRACKING_START} onward, for the same RTO-tagging "
    "reliability reason as above."
)

gcol1, gcol2 = st.columns([1, 3])
with gcol1:
    geo_level = st.radio("Group by", ["City", "State"], horizontal=True, key="delivery_geo_level")
geo_field = "Shipping City" if geo_level == "City" else "Shipping Province"

geo_scope = shipped_rto
if geo_level == "City" and "Shipping Province" in shipped_rto.columns:
    state_options = ["All States"] + sorted(shipped_rto["Shipping Province"].dropna().unique().tolist())
    sel_state = st.selectbox("Filter to State", state_options, key="delivery_geo_state")
    if sel_state != "All States":
        geo_scope = shipped_rto[shipped_rto["Shipping Province"] == sel_state]

min_orders = st.slider("Minimum shipped orders threshold", 5, 200, 30, key="delivery_min_orders")

col1, col2 = st.columns(2)
with col1:
    if geo_field in geo_scope.columns:
        geo = geo_scope.groupby(geo_field).agg(
            Shipped=("Is Shipped", "sum"),
            Delivered=("Is Delivered", "sum"),
        )
        geo = geo[geo["Shipped"] >= min_orders]
        geo["Delivery Rate %"] = (geo["Delivered"] / geo["Shipped"] * 100).round(2)
        best = geo.sort_values("Delivery Rate %", ascending=False).head(10).reset_index()
        title_suffix = f" in {sel_state}" if geo_level == "City" and sel_state != "All States" else ""
        fig = px.bar(best, x="Delivery Rate %", y=geo_field, orientation="h",
                      title=f"Best Delivery Rate {geo_level}s{title_suffix}", color_discrete_sequence=[SUCCESS])
        fig.update_layout(yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig, width='stretch')

with col2:
    if geo_field in geo_scope.columns:
        worst = geo.sort_values("Delivery Rate %", ascending=True).head(10).reset_index()
        fig = px.bar(worst, x="Delivery Rate %", y=geo_field, orientation="h",
                      title=f"Worst Delivery Rate {geo_level}s{title_suffix}", color_discrete_sequence=[DANGER])
        fig.update_layout(yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig, width='stretch')

    if geo.empty:
        st.info(f"No {geo_level.lower()}s meet the minimum shipped-orders threshold in this scope.")

In [ ]:
if "Shipping Province" in shipped_rto.columns:
    st.subheader("Delivery Rate by State")
    geo_state = shipped_rto.groupby("Shipping Province").agg(
        Shipped=("Is Shipped", "sum"),
        Delivered=("Is Delivered", "sum"),
    )
    geo_state = geo_state[geo_state["Shipped"] >= min_orders]
    geo_state["Delivery Rate %"] = (geo_state["Delivered"] / geo_state["Shipped"] * 100).round(2)
    geo_state["RTO Rate %"] = (100 - geo_state["Delivery Rate %"]).round(2)
    geo_state = geo_state.sort_values("Delivery Rate %", ascending=False)
    st.dataframe(geo_state, width='stretch')

In [ ]:
st.markdown("---")
st.subheader("Non-Delivery Diagnostic Report")
st.caption(
    "Your order export doesn't include an explicit 'reason for non-delivery' field "
    "(no NDR/courier-attempt feed). The breakdowns below use the fields that are "
    "available to show *what's correlated with* RTO -- useful as proxies for diagnosing "
    f"where non-deliveries come from. Also limited to orders shipped from {RTO_TRACKING_START} "
    "onward, for the same RTO-tagging reliability reason as above."
)

col1, col2 = st.columns(2)
with col1:
    if "Payment Type" in shipped_rto.columns:
        pay = shipped_rto.groupby("Payment Type")["Any RTO"].agg(["mean", "count"]).reset_index()
        pay["RTO Rate %"] = (pay["mean"] * 100).round(2)
        fig = px.bar(pay, x="Payment Type", y="RTO Rate %", title="RTO Rate by Payment Type",
                      color="Payment Type", color_discrete_sequence=PALETTE,
                      hover_data={"count": True})
        st.plotly_chart(fig, width='stretch')

with col2:
    if "Discount Category" in shipped_rto.columns:
        disc = shipped_rto.groupby("Discount Category")["Any RTO"].agg(["mean", "count"]).reset_index()
        disc = disc[disc["count"] >= 50]
        disc["RTO Rate %"] = (disc["mean"] * 100).round(2)
        disc = disc.sort_values("RTO Rate %", ascending=False)
        fig = px.bar(disc, x="RTO Rate %", y="Discount Category", orientation="h",
                      title="RTO Rate by Discount Category", color_discrete_sequence=[ACCENT1],
                      hover_data={"count": True})
        fig.update_layout(yaxis=dict(autorange="reversed"))
        st.plotly_chart(fig, width='stretch')

In [ ]:
col1, col2 = st.columns(2)
with col1:
    if "Total" in shipped_rto.columns:
        value_bins = shipped_rto.copy()
        value_bins["Order Value"] = pd.cut(
            value_bins["Total"], bins=[0, 500, 1000, 1500, 2000, 3000, float("inf")],
            labels=["0-500", "500-1k", "1k-1.5k", "1.5k-2k", "2k-3k", "3k+"]
        )
        vb = value_bins.groupby("Order Value", observed=True)["Any RTO"].agg(["mean", "count"]).reset_index()
        vb["RTO Rate %"] = (vb["mean"] * 100).round(2)
        fig = px.bar(vb, x="Order Value", y="RTO Rate %", title="RTO Rate by Order Value Bucket",
                      color_discrete_sequence=[ACCENT2], hover_data={"count": True})
        st.plotly_chart(fig, width='stretch')

with col2:
    if "Customer Key" in shipped_rto.columns and "Created at" in shipped_rto.columns:
        sh = shipped_rto.copy()
        sh["Order Rank"] = sh.groupby("Customer Key")["Created at"].rank(method="first")
        sh["Customer Type"] = sh["Order Rank"].eq(1).map({True: "First Order", False: "Repeat Order"})
        ct = sh.groupby("Customer Type")["Any RTO"].agg(["mean", "count"]).reset_index()
        ct["RTO Rate %"] = (ct["mean"] * 100).round(2)
        fig = px.bar(ct, x="Customer Type", y="RTO Rate %", title="RTO Rate: First vs Repeat Order",
                      color="Customer Type", color_discrete_sequence=PALETTE, hover_data={"count": True})
        st.plotly_chart(fig, width='stretch')

# --- Deeper split: cross First/Repeat with the customer's payment behavior, including
# customers who switch between COD and Prepaid across their orders. This surfaces
# "COD -> Prepaid" converts (customer started COD, later paid upfront) and
# "Prepaid -> COD" reversions, and how RTO risk differs for each. ---
if {"Customer Key", "Created at", "Payment Type", "Name"}.issubset(orders.columns):
    st.markdown("###")
    st.subheader("RTO Rate by Customer Payment Behavior")

    cust_orders = orders.dropna(subset=["Customer Key"]).sort_values("Created at")[
        ["Name", "Customer Key", "Payment Type"]
    ].copy()
    cust_orders["Order Seq"] = cust_orders.groupby("Customer Key").cumcount() + 1
    first_payment = cust_orders.loc[cust_orders["Order Seq"] == 1].set_index("Customer Key")["Payment Type"]
    cust_orders["First Payment Type"] = cust_orders["Customer Key"].map(first_payment)

    def _segment(row):
        if row["Order Seq"] == 1:
            return f"First Order ({row['Payment Type']})"
        if row["Payment Type"] == row["First Payment Type"]:
            return f"Repeat - Still {row['Payment Type']}"
        return f"Repeat - Switched {row['First Payment Type']} → {row['Payment Type']}"

    cust_orders["Segment"] = cust_orders.apply(_segment, axis=1)
    seg_map = cust_orders.set_index("Name")["Segment"]

    seg = shipped_rto.copy()
    seg["Segment"] = seg["Name"].map(seg_map)
    seg_summary = seg.groupby("Segment")["Any RTO"].agg(["mean", "count"]).reset_index()
    seg_summary = seg_summary[seg_summary["count"] >= 20]
    seg_summary["RTO Rate %"] = (seg_summary["mean"] * 100).round(2)
    seg_summary = seg_summary.sort_values("RTO Rate %", ascending=False)

    color_map = {
        "Repeat - Still COD": DANGER,
        "First Order (COD)": PALETTE[4],
        "Repeat - Switched Prepaid → COD": PALETTE[4],
        "Repeat - Switched COD → Prepaid": ACCENT1,
        "Repeat - Still Prepaid": SUCCESS,
        "First Order (Prepaid)": SUCCESS,
    }
    fig = px.bar(seg_summary, x="RTO Rate %", y="Segment", orientation="h",
                  title="RTO Rate by First-Order / Repeat-Order Payment Behavior",
                  color="Segment", color_discrete_map=color_map, hover_data={"count": True})
    fig.update_layout(yaxis=dict(autorange="reversed"), showlegend=False)
    st.plotly_chart(fig, width='stretch')

    cod_to_prepaid = seg_summary.loc[seg_summary["Segment"].str.contains("COD → Prepaid"), "RTO Rate %"]
    still_cod = seg_summary.loc[seg_summary["Segment"] == "Repeat - Still COD", "RTO Rate %"]
    prepaid_to_cod = seg_summary.loc[seg_summary["Segment"].str.contains("Prepaid → COD"), "RTO Rate %"]
    if not cod_to_prepaid.empty and not still_cod.empty:
        st.caption(
            f"Repeat customers who switch from COD to Prepaid return only **{cod_to_prepaid.values[0]:.1f}%** "
            f"of the time, vs **{still_cod.values[0]:.1f}%** for repeat customers who stay on COD -- and repeat "
            f"COD customers actually have a *higher* RTO rate than brand-new COD customers. "
            + (f"Going the other way, customers who switch from Prepaid back to COD spike to "
               f"**{prepaid_to_cod.values[0]:.1f}%** RTO. " if not prepaid_to_cod.empty else "")
            + "Getting repeat COD buyers to pay upfront (discounts, COD verification calls, "
              "loyalty prepaid offers) looks like one of the highest-leverage RTO levers available."
        )

In [ ]:
# COD verification status, if present (Tags contain releasit_cod_form / COD-Verified / COD-Pending)
if "Tags" in shipped_rto.columns:
    tags_cod = shipped_rto["Tags"].fillna("").astype(str)

    def _cod_group(t):
        if "COD-Verified" in t or "Order Verified" in t:
            return "COD Verified"
        if "COD-Pending" in t:
            return "COD Pending / Unverified"
        return "Other"

    cod_df = pd.DataFrame({"COD Verification": [_cod_group(t) for t in tags_cod], "Any RTO": shipped_rto["Any RTO"].values})
    cod_summary = cod_df.groupby("COD Verification")["Any RTO"].agg(["mean", "count"]).reset_index()
    cod_summary["RTO Rate %"] = (cod_summary["mean"] * 100).round(2)
    if cod_summary.loc[cod_summary["COD Verification"] != "Other", "count"].sum() >= 50:
        st.markdown("###")
        fig = px.bar(cod_summary, x="COD Verification", y="RTO Rate %", title="RTO Rate by COD Verification Status",
                      color="COD Verification",
                      color_discrete_map={"COD Verified": SUCCESS, "COD Pending / Unverified": DANGER, "Other": PALETTE[4]},
                      hover_data={"count": True})
        st.plotly_chart(fig, width="stretch")
        verified_rate = cod_summary.loc[cod_summary["COD Verification"] == "COD Verified", "RTO Rate %"]
        pending_rate = cod_summary.loc[cod_summary["COD Verification"] == "COD Pending / Unverified", "RTO Rate %"]
        if not verified_rate.empty and not pending_rate.empty:
            st.caption(
                f"Orders with an unverified/pending COD confirmation return at "
                f"**{pending_rate.values[0]:.0f}%**, vs **{verified_rate.values[0]:.0f}%** for orders where COD "
                "was confirmed via the verification form/WhatsApp/call. Prioritizing COD verification calls "
                "for unconfirmed orders is one of the highest-leverage levers to cut RTO."
            )

In [ ]:
# rto_prediction tag (courier/app risk flag), if present
if "Tags" in shipped_rto.columns:
    tags = shipped_rto["Tags"].astype(str)
    flagged = tags.str.contains("rto_prediction_high", case=False, na=False) | \
        tags.str.contains("rto_prediction_very-high", case=False, na=False)
    if flagged.sum() >= 20:
        flag_df = pd.DataFrame({"Flagged High RTO Risk": flagged, "Any RTO": shipped_rto["Any RTO"]})
        risk_summary = flag_df.groupby("Flagged High RTO Risk")["Any RTO"].agg(["mean", "count"]).reset_index()
        risk_summary["RTO Rate %"] = (risk_summary["mean"] * 100).round(2)
        risk_summary["Flagged High RTO Risk"] = risk_summary["Flagged High RTO Risk"].map({True: "Flagged High Risk", False: "Not Flagged"})
        st.markdown("###")
        fig = px.bar(risk_summary, x="Flagged High RTO Risk", y="RTO Rate %",
                      title="Actual RTO Rate vs Courier 'High RTO Risk' Flag", color="Flagged High RTO Risk",
                      color_discrete_map={"Flagged High Risk": DANGER, "Not Flagged": SUCCESS},
                      hover_data={"count": True})
        st.plotly_chart(fig, width='stretch')
        st.caption(
            f"{int(flagged.sum()):,} shipped orders are tagged by the courier/app as high RTO "
            f"risk -- these orders return at {risk_summary.loc[risk_summary['Flagged High RTO Risk'] == 'Flagged High Risk', 'RTO Rate %'].values[0]:.0f}% "
            "vs the baseline. Worth a manual confirmation call before shipping these."
        )